In [ ]:
import os

MODEL_NAME       = "osmdet"
MODEL_CHECKPOINT = "tarryzhang/OSM-Det"
BATCH_SIZE       = 1
MAX_LENGTH       = 4096

DATA_DIR = "../../../data"
TRAIN_DATA_PATH  = f"{DATA_DIR}/train_data.parquet"
TEST_DATA_PATH   = f"{DATA_DIR}/test_data.parquet"
MULTISOCIAL_PATH = f"{DATA_DIR}/external_data/multisocial/multisocial_anonymized.csv"
AIGTBENCH_PATH   = f"{DATA_DIR}/external_data/AIGTBench"
FOX8_PATH        = f"{DATA_DIR}/external_data/fox8_23_dataset.ndjson"

AIGTBENCH_PLATFORM_FILTER = "reddit"

SCORES_DIR = "../../../scores"
os.makedirs(SCORES_DIR, exist_ok=True)
print(f"Scores will be saved to: {os.path.abspath(SCORES_DIR)}")

In [ ]:
import json
import joblib

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_CHECKPOINT)
model.to("cuda")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
print("Model loaded.")

In [ ]:
def detect_ai_text_batch(texts: list, batch_size: int = BATCH_SIZE, max_length: int = MAX_LENGTH) -> dict:
    unique_texts = list(set(texts))
    lengths = [len(t) for t in unique_texts]
    order = np.argsort(lengths)[::-1]
    sorted_texts = [unique_texts[i] for i in order]

    results = {}
    for i in tqdm(range(0, len(sorted_texts), batch_size)):
        batch = sorted_texts[i : i + batch_size]
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            max_length=max_length,
            truncation=True,
            padding=True,
        ).to("cuda")
        with torch.no_grad():
            logits = model(**inputs).logits
            probs = torch.nn.functional.softmax(logits, dim=-1)
        results.update({text: prob[1].item() for text, prob in zip(batch, probs)})
    return results


def extract_pair_scores(df, scores_dict: dict) -> dict:
    result = {}
    for _, row in df.iterrows():
        key = (row["model"], row["pipeline"], row["user_id"], row["thread_id"])
        result[key] = (
            scores_dict.get(row["real_text"], float("nan")),
            scores_dict.get(row["generated_text"], float("nan")),
        )
    return result

## Load Datasets

In [ ]:
train = pd.read_parquet(TRAIN_DATA_PATH)
test  = pd.read_parquet(TEST_DATA_PATH)
val   = train[train.is_val == True].copy()

print(f"Train rows: {len(train):,}")
print(f"Val rows:   {len(val):,}")
print(f"Test rows:  {len(test):,}")

In [ ]:
multisocial = pd.read_csv(MULTISOCIAL_PATH)
multisocial = multisocial[multisocial.split == "test"].copy()
print(f"Multisocial test rows: {len(multisocial):,}")

In [ ]:
aigtbench = pd.read_parquet(AIGTBENCH_PATH)
if AIGTBENCH_PLATFORM_FILTER:
    aigtbench = aigtbench[aigtbench.social_media_platform == AIGTBENCH_PLATFORM_FILTER].copy()
print(f"AIGTBench rows (filter={AIGTBENCH_PLATFORM_FILTER!r}): {len(aigtbench):,}")

In [ ]:
with open(FOX8_PATH, "r", encoding="utf-8") as f:
    fox8_raw = [json.loads(line) for line in f]

fox8_rows = []
for sample in tqdm(fox8_raw, desc="Parsing fox8"):
    label   = 0 if sample["label"] == "human" else 1
    user_id = sample["user_id"]
    for tweet in sample["user_tweets"]:
        fox8_rows.append({"text": tweet["text"], "user_id": user_id, "label": label})

fox8 = pd.DataFrame(fox8_rows)
print(f"Fox8 rows: {len(fox8):,}  |  users: {fox8.user_id.nunique():,}")

## Score Our Paired Dataset (Val + Test)

In [ ]:
all_our_texts = (
    list(val["real_text"]) + list(val["generated_text"]) +
    list(test["real_text"]) + list(test["generated_text"])
)
print(f"Unique texts to score (val+test): {len(set(all_our_texts)):,}")

our_scores = detect_ai_text_batch(all_our_texts)

val_scores  = extract_pair_scores(val,  our_scores)
test_scores = extract_pair_scores(test, our_scores)

joblib.dump(test_scores, f"{SCORES_DIR}/{MODEL_NAME}_test_scores.joblib")
joblib.dump(val_scores,  f"{SCORES_DIR}/{MODEL_NAME}_val_scores.joblib")
print(f"Saved {len(test_scores):,} test pairs and {len(val_scores):,} val pairs.")

## Score External Benchmarks

In [ ]:
print("Scoring Multisocial...")
ms_dict = detect_ai_text_batch(multisocial["text"].tolist())
multisocial[f"score_{MODEL_NAME}"] = multisocial["text"].map(ms_dict)
print("Done.")

In [ ]:
print("Scoring AIGTBench...")
aigt_dict = detect_ai_text_batch(aigtbench["text"].tolist())
aigtbench[f"score_{MODEL_NAME}"] = aigtbench["text"].map(aigt_dict)
print("Done.")

In [ ]:
print("Scoring Fox8-23...")
fox8_dict = detect_ai_text_batch(fox8["text"].tolist())
fox8[f"score_{MODEL_NAME}"] = fox8["text"].map(fox8_dict)
print("Done.")

## Save External Benchmark Scores

In [ ]:
score_col = f"score_{MODEL_NAME}"

ms_out   = f"{SCORES_DIR}/multisocial_scores_{MODEL_NAME}.parquet"
aigt_out = f"{SCORES_DIR}/aigtbench_scores_{MODEL_NAME}.parquet"
fox8_out = f"{SCORES_DIR}/fox8_scores_{MODEL_NAME}.parquet"

multisocial[["text", "label", "language", "source", score_col]].to_parquet(ms_out, index=True)
aigtbench[["text", "label", score_col]].to_parquet(aigt_out, index=True)
fox8[["text", "user_id", "label", score_col]].to_parquet(fox8_out, index=True)

print(f"Multisocial scores: {ms_out}")
print(f"AIGTBench scores:   {aigt_out}")
print(f"Fox8-23 scores:     {fox8_out}")

## Evaluation Summary

In [ ]:
score_col = f"score_{MODEL_NAME}"

auc_ms   = roc_auc_score(multisocial["label"], multisocial[score_col])
auc_aigt = roc_auc_score(aigtbench["label"],   aigtbench[score_col])

auc_fox8_text = roc_auc_score(fox8["label"], fox8[score_col])
fox8_agg      = fox8.groupby("user_id")[[score_col, "label"]].mean().reset_index()
auc_fox8_user = roc_auc_score(fox8_agg["label"].round().astype(int), fox8_agg[score_col])

results_df = pd.DataFrame([{
    "model":         MODEL_NAME,
    "multisocial":   round(auc_ms, 4),
    "aigtbench":     round(auc_aigt, 4),
    "fox8_text":     round(auc_fox8_text, 4),
    "fox8_user_agg": round(auc_fox8_user, 4),
}]).set_index("model")

print("ROC-AUC Summary")
results_df